In [ ]:
import torch
import torch.nn as nn

class Generator(nn.Module):
    def __init__(self, ngpu, nc=1, nz=100, ngf=64):
        super(Generator, self).__init__()
        self.ngpu = ngpu
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d(     nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # state size. (ngf*8) x 4 x 4
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # state size. (ngf*4) x 8 x 8
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # state size. (ngf*2) x 16 x 16
            nn.ConvTranspose2d(ngf * 2,     ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(    ngf,      nc, kernel_size=1, stride=1, padding=2, bias=False),
            nn.Tanh()
        )

    def forward(self, input):
        if input.is_cuda and self.ngpu > 1:
            output = nn.parallel.data_parallel(self.main, input, range(self.ngpu))
        else:
            output = self.main(input)
        return output


class Discriminator(nn.Module):
    def __init__(self, ngpu, nc=1, ndf=64):
        super(Discriminator, self).__init__()
        self.ngpu = ngpu
        self.main = nn.Sequential(
            # input is (nc) x 64 x 64
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf) x 32 x 32
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf*2) x 16 x 16
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf*4) x 8 x 8
            nn.Conv2d(ndf * 4, 1, 4, 2, 1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, input):
        if input.is_cuda and self.ngpu > 1:
            output = nn.parallel.data_parallel(self.main, input, range(self.ngpu))
        else:
            output = self.main(input)
        return output.view(-1, 1).squeeze(1)

In [ ]:
from google.colab import drive
drive.mount('/content/drive/Shareddrives/run4(test1)')

# df = pd.read_csv('/content/drive/Shareddrives/run4(test1)')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
!gsutil cp -r gs://data_from_runs/run4/model/netG* ./run4/


In [ ]:
!gsutil cp -r gs://data_from_runs/test3/model/netG* ./test3/


In [ ]:
!gsutil cp -r gs://data_from_runs/test2/model/netG* ./test2/


In [ ]:
!gsutil cp -r gs://data_from_runs/2e-4x2l0l02warmstartsynth/model/netG* ./2e-4x2l0l02warmstartsynth/model/
!wget https://github.com/csinva/gan-vae-pretrained-pytorch/raw/master/mnist_dcgan/weights/netG_epoch_99.pth -O 2e-4x2l0l02warmstartsynth/model/netG_epoch_-1.pth

In [ ]:

models=[]
generator = Generator(ngpu=1, nc=1, nz=100, ngf=64)
generator.load_state_dict(torch.load(f"2e-4x2l0l02warmstartsynth/model/netG_epoch_0000.pth"))
generator.to(device)
models.append(generator)
for i in range(500):
  generator = Generator(ngpu=1, nc=1, nz=100, ngf=64)
  generator.load_state_dict(torch.load(f"2e-4x2l0l02warmstartsynth/model/netG_epoch_{i:03d}.pth"))
  generator.to(device)
  models.append(generator)

In [ ]:

models_retrain_fake=[]
# generator = Generator(ngpu=1, nc=1, nz=100, ngf=64)
# generator.load_state_dict(torch.load(f"2e-4x2l0l02warmstartsynth/model/netG_epoch_0000.pth"))
# generator.to(device)
# models_retrain_fake.append(generator)
for i in range(100):
  generator = Generator(ngpu=1, nc=1, nz=100, ngf=64)
  generator.load_state_dict(torch.load(f"run4/netG_epoch_{i:03d}.pth"))
  generator.to(device)
  models_retrain_fake.append(generator)

In [ ]:

models_posttrain_real=[]
generator = Generator(ngpu=1, nc=1, nz=100, ngf=64)
generator.load_state_dict(torch.load(f"2e-4x2l0l02warmstartsynth/model/netG_epoch_0000.pth"))
generator.to(device)
models_posttrain_real.append(generator)
for i in range(100):
  generator = Generator(ngpu=1, nc=1, nz=100, ngf=64)
  generator.load_state_dict(torch.load(f"test2/netG_epoch_{i:03d}.pth"))
  generator.to(device)
  models_posttrain_real.append(generator)

In [ ]:

models_retrain_real=[]
# generator = Generator(ngpu=1, nc=1, nz=100, ngf=64)
# generator.load_state_dict(torch.load(f"2e-4x2l0l02warmstartsynth/model/netG_epoch_0000.pth"))
# generator.to(device)
# models_retrain_real.append(generator)
for i in range(100):
  generator = Generator(ngpu=1, nc=1, nz=100, ngf=64)
  generator.load_state_dict(torch.load(f"test3/netG_epoch_{i:03d}.pth"))
  generator.to(device)
  models_retrain_real.append(generator)

In [ ]:
!pip install torchmetrics
!pip install torch-fidelity

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import MNIST
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Subset
# import torch.ignite.metrics.FID as FID
from torchmetrics.image.fid import FrechetInceptionDistance
fid = FrechetInceptionDistance(feature=2048,reset_real_features=False).to(device)
transform = transforms.Compose([
    transforms.ToTensor(),
])

real_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# Find indices of images that are not zero
# non_one_indices = [i for i, (img, label) in enumerate(real_dataset) if label != 1]

# # Create a subset of the dataset with only non-zero images
# real_dataset = Subset(real_dataset, non_one_indices)

real_loader = DataLoader(real_dataset, batch_size=64, shuffle=True)


In [ ]:

for real_images,i in real_loader:
    real_images = real_images.to(device)
    real_images = (real_images * 255).to(torch.uint8)

    # FID expects 3-channel images, so repeat grayscale to RGB
    real_images = real_images.repeat(1, 3, 1, 1)
    fid.update(real_images, real=True)



In [ ]:
fids=[]
with torch.no_grad():
  nz=100
  for i,model in enumerate(models_retrain_real):
    print(f"model {i}")
    for i in range(100):
      generated_imgs = model(torch.randn(32,100,1,1).to(device))

      # generated_imgs = real_images.to(device)
      generated_imgs = (generated_imgs * 255).to(torch.uint8)
      generated_imgs = generated_imgs.repeat(1, 3, 1, 1)
      fid.update(generated_imgs, real=False)
    fid_score = fid.compute().cpu().numpy()
    fids.append(fid_score)
    print(f"FID Score: {fid_score}")
    fid.reset()

In [ ]:
import matplotlib.pyplot as plt
plt.plot(range(101),fids)
plt.title("FID Score vs Epoch for retrain from scratch, artificial data")
plt.xlabel("Epoch")
plt.ylabel("FID Score")

In [ ]:
fids[12]

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.models as models
from torchvision import datasets, transforms

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load a pretrained model (ResNet18)
classifier = models.resnet18(pretrained=True)

# Modify first conv layer to accept 1 channel (grayscale) instead of 3 (RGB)
classifier.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

# Modify the last layer for MNIST (10 classes)
classifier.fc = nn.Linear(512, 10)
classifier = classifier.to(device)

# Transform: resize to 224x224 (ResNet input size) and normalize
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Load MNIST dataset
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Create data loaders
train_loader = DataLoader(mnist_train, batch_size=64, shuffle=True)
test_loader = DataLoader(mnist_test, batch_size=64, shuffle=False)


# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(classifier.parameters(), lr=0.001)

# Training loop
num_epochs = 3

for epoch in range(num_epochs):
    classifier.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = classifier(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()

        if batch_idx % 200 == 0:
            print(f'Epoch {epoch+1}/{num_epochs}, Batch {batch_idx}/{len(train_loader)}, '
                  f'Loss: {loss.item():.4f}, Acc: {100.*correct/total:.2f}%')

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    print(f'Epoch {epoch+1} completed - Avg Loss: {epoch_loss:.4f}, Training Acc: {epoch_acc:.2f}%')

# Evaluation
classifier.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = classifier(data)
        _, predicted = output.max(1)
        test_total += target.size(0)
        test_correct += predicted.eq(target).sum().item()

test_acc = 100. * test_correct / test_total
print(f'\nTest Accuracy: {test_acc:.2f}%')

In [ ]:
# Evaluation
classifier.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = classifier(data)
        print(output)
        _, predicted = output.max(1)
        test_total += target.size(0)
        test_correct += predicted.eq(target).sum().item()

test_acc = 100. * test_correct / test_total
print(f'\nTest Accuracy: {test_acc:.2f}%')

In [ ]:
from collections import Counter
import numpy as np
# Transform: resize to 224x224 (ResNet input size) and normalize
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Normalize((0.1307,), (0.3081,))
])

avg_certainties=[]
max_classes_epochs=[]
pct_ones = []
n_batches = 100
with torch.no_grad():
  nz=100
  for i,model in enumerate(models_retrain_real):
    max_classes=[0 for i in range(10)]
    certainties=[]
    print(f"model {i}")
    for i in range(n_batches):
      generated_imgs = model(torch.randn(32,100,1,1).to(device))
      generated_imgs = transform(generated_imgs)
      classification = torch.softmax(classifier(generated_imgs),dim=1)
      classification=classification.cpu().detach().numpy()
      # print(classification)
      max_class = classification.argmax(axis=1)
      for cls in max_class:
        max_classes[cls]+=1
      certainty = classification.max(axis=1)
      certainties.extend(certainty)
    max_classes_epochs.append(max_classes)
    pct_ones.append(max_classes[1]/(32*n_batches))
    avg_certainties.append(np.mean(certainties))

      # print(max_class)

In [ ]:
list(enumerate(pct_ones))

In [ ]:
pct_ones[0]

In [ ]:
fids[14]

In [ ]:
plt.plot(range(50),pct_ones)
plt.xlabel("Epoch")
plt.ylabel("Proportion of ones")
plt.title("Proportion of ones per epoch for retrained model, artificial data")
plt.show()

In [ ]:
for x in [0]+list(range(2,10)):
  pct_xs = [m[x]/(32*100) for m in max_classes_epochs]
  plt.plot(range(100),pct_xs,label=f"class {x}")

In [ ]:
max_classes_epochs[11]

In [ ]:
plt.bar(range(10),[x/3200 for x in max_classes_epochs[0]])
plt.xlabel("Class")
plt.ylabel("Proportion of images")
plt.title("Proportion of images per class for pretrained model")
plt.show()

In [ ]:
plt.bar(range(10),[x/3200 for x in max_classes_epochs[50]])
plt.xlabel("Class")
plt.ylabel("Proportion of images")
plt.title("Proportion of images per class")
plt.show()

In [ ]:
# prompt: add the label for each digit to the graph

for x in range(10):
  pct_xs = [m[x]/(32*100) for m in max_classes_epochs]
  plt.plot(range(50),pct_xs,label=f"class {x}")
plt.xlabel("Epoch")
plt.ylabel("Proportion of digits")
plt.title("Proportion of digits per epoch")
plt.legend()
plt.show()

In [ ]:
from collections import Counter
counter = Counter()
for imgs,target in real_loader:
  # print(imgs.shape)
  # print(target)
  # counter.update(target.numpy())
# print(counter)
  imgs=imgs.to(device)
  print(classifier(imgs))

In [ ]:
np.save("fids.npy",np.array(fids))

In [ ]:
np.save("pct_ones.npy",np.array(pct_ones))

In [ ]:
np.save("max_classes_epochs.npy",np.array(max_classes_epochs))

In [ ]:

class Encoder(nn.Module):
  class AddGaussianNoise(nn.Module):
    def __init__(self, mean=0.0, std=0.05):
        super().__init__()
        self.mean = mean
        self.std = std

    def forward(self, x):
        if not self.training:
            return x
        return x + torch.randn_like(x) * self.std + self.mean

  def __init__(self):
    super().__init__()

    self.noise = self.AddGaussianNoise(0.0, 0.1)

    self.feature_extractor = nn.Sequential(
      nn.Conv2d(1, 32, 4, stride=2, padding=1),
      nn.ReLU(),
      nn.Conv2d(32, 64, 4, stride=2, padding=1),
      nn.ReLU(),
      nn.Conv2d(64, 128, 3, stride=1, padding=0),
      nn.ReLU()
    )

    self.pool = nn.AdaptiveAvgPool2d((1, 1))
    self.fc = nn.Linear(128, 32)

  def forward(self, x):
    if self.training:
      x = self.noise(x)

    x = self.feature_extractor(x)
    x = self.pool(x)
    x = x.view(x.size(0), -1)
    return self.fc(x)
svdd_encoder = Encoder().to(device)
svdd_encoder.load_state_dict(torch.load("mnist_svdd.pth"))
center = torch.Tensor(np.load("center.npy")).to(device)
threshold = torch.Tensor(np.load("threshold.npy")).to(device)
def classify_svdd(x):
  y= torch.sum((svdd_encoder(x)-center)**2)
  # print(y)
  return y>-threshold

In [ ]:

class AddGaussianNoise(object):
    def __init__(self, mean=0.0, std=0.05):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        return tensor + torch.randn_like(tensor) * self.std + self.mean

transform_svdd = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(0, translate=(0.05,0.05)),
    AddGaussianNoise(0.0, 0.1)
])



In [ ]:

# Transform: resize to 224x224 (ResNet input size) and normalize
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [ ]:
print(threshold)

In [ ]:
real=0
with torch.no_grad():
  classes = [0 for i in range(10)]
  for i in range(n_batches):
    generated_imgs = models[0](torch.randn(32,100,1,1).to(device))
    for img in generated_imgs:
      img= img.unsqueeze(0)
      is_real = classify_svdd(transform_svdd(img))

      if is_real:
        real+=1
        classes[classifier(transform(img)).argmax(dim=1)] += 1
  print(real/3200)
  print(classes)


In [ ]:
from torch.utils.data import Dataset

class GeneratedDataset(Dataset):
  def __init__(self, images, transform=None):
    self.images = images
    self.transform = transform
  def __len__(self):
    return len(self.images)
  def __getitem__(self, idx):
    img = self.images[idx]
    img = img.squeeze(0).numpy()
    img = Image.fromarray((img * 255).astype('uint8'))
    if self.transform:
      img = self.transform(img)
    return img

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import numpy as np

num_samples = 60000
batch_size = 1024
nz = 100

all_images = []
with torch.no_grad():
  for _ in range((num_samples + batch_size - 1) // batch_size):
    current_batch_size = min(batch_size, num_samples - len(all_images))
    z = torch.randn(current_batch_size, nz, 1, 1, device=device)
    fake_images = netG(z)
    all_images.append(fake_images.cpu())

all_images = torch.cat(all_images, dim=0)

transform = transforms.Compose([
    transforms.ToTensor(),
])
super_test_dataset = GeneratedDataset(all_images, transform=transform)
super_test_loader = DataLoader(super_test_dataset, batch_size=batch_size, shuffle=False)

print(f"Super test loader ready with {len(super_test_loader)} batches")

def anomaly_scores(loader):
  scores = []
  svdd.eval()

  with torch.no_grad():
    for x in loader:
      x = x.to(device)
      z = svdd(x)
      dist = torch.sum((z - c)**2, dim=1)

      scores.extend(dist.cpu().numpy())

  return np.array(scores)

scores = anomaly_scores(super_test_loader)

In [ ]:
class GeneratedDatasetFiltered(Dataset):
  def __init__(self, images, transform=None):
    self.images = images
    self.transform = transform
  def __len__(self):
    return len(self.images)
  def __getitem__(self, idx):
    img = self.images[idx]
    return img

In [ ]:
filtered_images = all_images[~pred_is_1]
print(f"Filtered images: {filtered_images.shape[0]} kept out of {all_images.shape[0]}")

filtered_dataset = GeneratedDatasetFiltered(filtered_images, transform=transform)
filtered_loader = DataLoader(
    filtered_dataset,
    batch_size=batch_size,
    shuffle=False
)
